# maxpool-reduce composite — cx17: MaxPool2d with stride: as_strided windowing + einops.reduce(max)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `maxpool-reduce`, `as-strided-windowing`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "maxpool-reduce"
DD_ATOM_IDS = ["maxpool-reduce", "as-strided-windowing"]
DD_SUBTOPICS = ["CNN: MaxPool as reduce", "PyTorch: as_strided windowing"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

**Non-overlapping MaxPool** (kernel==stride) is trivially `einops.reduce(x, 'b c (h p1) (w p2) -> b c h w', 'max')` — the parenthesized axis factoring does both the windowing and the reduction in one expression.

**Overlapping MaxPool** (kernel ≠ stride — the ResNet stem case) needs the two atoms separately:

1. **`as-strided-windowing`** — build a `(B, C, OH, OW, KH, KW)` view of the input where each `(KH, KW)` patch is one pool window. With pool stride `S`, the OH/OW strides are `s_h * S, s_w * S`; the KH/KW strides are `s_h, s_w` (within-window walks element by element).
2. **`maxpool-reduce`** — apply `einops.reduce(..., 'b c oh ow kh kw -> b c oh ow', 'max')` to collapse the `(KH, KW)` pool patch into one scalar (its max).

**Why this composition matters.** Once you can pool with strided windowing + `reduce(max)`, you can change one letter (`'max' -> 'mean'`) to get AvgPool (see cx18). The window view is shared infrastructure — it's the same trick conv uses.

**Edge: kernel == stride.** Strides collapse to `(s_h * K, s_w * K, s_h, s_w)` — the non-overlapping case. The composite handles both because the formula is the same.

### Composite Exercise — MaxPool2d with stride: as_strided windowing + einops.reduce(max)

**Atoms exercised together**: `maxpool-reduce`, `as-strided-windowing`

Implement `cx17_maxpool2d(x, kernel_size, stride=None)`.

- `x`: float tensor `(B, C, H, W)`.
- `kernel_size`: int (square kernel) — the pool window size.
- `stride`: int or None. If `None`, defaults to `kernel_size` (non-overlapping, PyTorch's default). May be smaller than `kernel_size` (overlapping pool).
- Return: tensor `(B, C, OH, OW)` matching `F.max_pool2d(x, kernel_size, stride=stride)` (no padding).

Use `as_strided` to build the window view, then `einops.reduce` with `'max'` to pool. Compute `OH = (H - K) // S + 1`, `OW = (W - K) // S + 1`.

The test fuzzes overlapping and non-overlapping cases against `F.max_pool2d`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx17_maxpool2d(x, kernel_size, stride=None):
    raise NotImplementedError

def _test_cx17():
    from torch.nn import functional as F
    rng = t.Generator().manual_seed(17)

    # Case A: hand example, non-overlapping.
    x = t.tensor([[[
        [1.0, 2.0, 3.0, 4.0],
        [5.0, 6.0, 7.0, 8.0],
        [9.0, 1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0, 7.0],
    ]]])
    y = cx17_maxpool2d(x, kernel_size=2)
    assert tuple(y.shape) == (1, 1, 2, 2), f'shape: {tuple(y.shape)}'
    assert t.allclose(y, t.tensor([[[[6.0, 8.0], [9.0, 7.0]]]]))

    # Case B: overlapping (ResNet stem case) — kernel=3, stride=2.
    for B,C,H,W,K,S in [(2,3,8,8,3,2),(1,1,16,16,3,2),(2,4,10,12,2,1),(1,2,7,7,3,2)]:
        xr = t.randn(B, C, H, W, generator=rng)
        yours = cx17_maxpool2d(xr, kernel_size=K, stride=S)
        yref = F.max_pool2d(xr, kernel_size=K, stride=S)
        assert tuple(yours.shape) == tuple(yref.shape), f'shape diff on {(B,C,H,W,K,S)}'
        assert t.allclose(yours, yref, atol=1e-5), f'value diff on {(B,C,H,W,K,S)}'

    # Case C: default stride (None) → non-overlapping pool.
    xr = t.randn(2, 3, 8, 8, generator=rng)
    yours = cx17_maxpool2d(xr, kernel_size=4)
    yref = F.max_pool2d(xr, kernel_size=4)
    assert tuple(yours.shape) == (2, 3, 2, 2)
    assert t.allclose(yours, yref, atol=1e-5)

    # Case D: K=1 → identity (max over a 1x1 window is the value itself).
    xr = t.randn(1, 2, 4, 4, generator=rng)
    assert t.allclose(cx17_maxpool2d(xr, kernel_size=1, stride=1), xr, atol=1e-7)
    _dd_passed.add('cx17')

_test_cx17()

<details><summary>Show solution — cx17</summary>

```python
def cx17_maxpool2d(x, kernel_size, stride=None):
    K = kernel_size
    S = K if stride is None else stride
    B, C, H, W = x.shape
    OH = (H - K) // S + 1
    OW = (W - K) // S + 1
    # Atom A (as-strided-windowing): step S on OH/OW, step 1 on KH/KW.
    s_b, s_c, s_h, s_w = x.stride()
    x_win = x.as_strided(
        size=(B, C, OH, OW, K, K),
        stride=(s_b, s_c, s_h * S, s_w * S, s_h, s_w),
    )
    # Atom B (maxpool-reduce): collapse the within-window axes by max.
    return einops.reduce(x_win, 'b c oh ow kh kw -> b c oh ow', 'max')
```

Notice that switching from MaxPool to AvgPool is a one-token change (`'max' -> 'mean'`). The windowing infrastructure is identical. That's the whole point of factoring pool as `window + reduce` — the reducer is a parameter.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx17'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx17',
        'subtopics': ["CNN: MaxPool as reduce", "PyTorch: as_strided windowing"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()